# 00 - Workflow overview

This notebook walks through the five PFC3D scripts in the order they run. Every code block below is **read from the
repository files**, so what you see is exactly what runs in PFC3D. Nothing here needs a PFC license.

Run the cells from the top. Each code cell prints a short excerpt, and the text above it says what to look for.

```
00_Main_Run  ->  01_Geometry_Material  ->  02_Foliation_SJM  ->  03_Confinement_Servo  ->  04_Axial_Loading
```

Each script saves the model state at its end (`.sav`), and the next one restores it. This is why you can repeat only
the last stages (for example a new friction value) without building the sample again.

In [1]:
from pathlib import Path
SCRIPTS = Path("..") / "01_Scripts"

def show(name, first, last):
    """Print lines first..last of a PFC script exactly as they are in the repository."""
    lines = (SCRIPTS / name).read_bytes().decode("utf-8", "replace").replace("\r", "").split("\n")
    for n in range(first, last + 1):
        print(f"{n:4d}  {lines[n-1]}")

## Step 0 - `00_Main_Run.dat`: parameters and call order

All global parameters live here. The specimen id (`amostra`) selects a `<id>_params.dat` file that overrides the
defaults (confining stress, dip, plane cohesion and friction). The last lines `call` the four stages in sequence.

Block 1 below: the numerical sample and the foliation parameters.

In [2]:
show("00_Main_Run.dat", 67, 92)

  67      ; --- amostra numerica ---
  68      ; Raios 1.0-2.0 mm tem Custo ~3x mais caro que 1.5-2.25 mm; para calibrar
  69      ; ****vale usar 1.5-2.25 mm e refinar so na versao final.
  70      global raio_min       = 0.00150     ; m
  71      global raio_max       = 0.00225     ; m
  72      global poro_ini       = 0.30        ; [v2] a partir de 0.25 ocorre muita sobreposicao (ball distribute) 
  73      global semente        = 10001       ; Define qual o seed
  74      global gap_bond       = 0.0002      ; m
  75      global p_fjkn         = 6.8e12      ; [v3] Pa/m (seguem matriz); antes 1.0e13 (E do modelo ~38 GPa x lab 25.4 GPa)
  76      global p_fjks         = 2.72e12     ; [v3] Pa/m (segue matriz); antes 4.0e12; mantida a razao ks/kn = 0.4
  77  
  78      ; --- foliacao (smooth joint) ---
  79      global sj_band        = 0.002       ; m; meia espessura da faixa de contatos do plano
  80      global p_sjcoh        = 12.0e6      ; [v3.4.1] Pa; so o padrao: o <amostra>_param

Things to notice:

- `p_fjkn` and `p_fjks` are the normal and shear stiffness of the **matrix** contacts (flat joint). The foliation plane reuses the same stiffness.
- `sj_band` is the half-thickness of the band of contacts that become the plane.
- `p_sjcoh` and `p_sjfa` are only defaults. The params file of each specimen replaces them.
- `p_sjten` (tensile strength of the plane) has no test data. It is a free choice.

## Step 1 - `01_Geometry_Material.dat`: sample and matrix

The script builds a cylinder and two platens (wall boxes), fills the cylinder with random spheres, and shrinks the
radii with simple linear contacts until the initial overlap stress is small. Then it installs the **flat joint** model
on all ball-ball contacts and bonds them.

Block 2: the flat joint installation (stage B of the script).

In [3]:
show("01_Geometry_Material.dat", 187, 200)

 187  ; =========================================================
 188  ; [v2] ETAPA B: INSTALA O FLAT JOINT E COLA
 189  ; =========================================================
 190  contact cmat add 1 model flatjoint property fj_kn [p_fjkn] fj_ks [p_fjks] fj_coh 45.0e6 fj_ten 12.0e6 fj_fa 32.0 fj_fric 0.6 range contact type 'ball-ball'
 191  ; Distancia de deteccao = gap de colagem
 192  contact cmat proximity [gap_bond]
 193  model clean
 194  contact cmat apply
 195  
 196  ; Acomoda com o flat joint ainda descolado
 197  model cycle 1000 calm 50
 198  model solve ratio-average 1e-3 cycles 30000
 199  model mechanical timestep auto
 200  model calm


`fj_kn` and `fj_ks` come from Main. The strength values (`fj_coh`, `fj_ten`, `fj_fa`) are written in this line and
were **not calibrated** against matrix tests (see the README, section Limitations).

## Step 2 - `02_Foliation_SJM.dat`: the foliation plane

Contacts within `sj_band` of a plane through the specimen centre receive the **smooth-joint** model. This model has no
particle roughness, so sliding follows the plane.

Block 3: select the contacts, switch the model, set the properties, bond.

In [4]:
show("02_Foliation_SJM.dat", 8, 26)

   8      global folia_dip  = foliacao_dip
   9      global folia_ddir = 90.0
  10      io.out('Foliacao: mergulho = ' + string(foliacao_dip) + ' graus')
  11      ; [v3.4.1] c e phi vem do params.dat da amostra; o processa_resultados.py le esta linha do log
  12      io.out('Parametros do plano aplicados: c = ' + string(p_sjcoh / 1.0e6) + ' MPa   phi = ' + string(p_sjfa) + ' graus   atrito residual (sj_fric) = ' + string(p_sjfric))
  13  end
  14  [calc_foliation_orientation]
  15  
  16  ; Grupo do plano por faixa de distancia
  17  contact group 'grupo_foliacao' range contact type 'ball-ball' plane dip [folia_dip] dip-direction [folia_ddir] origin (0.0, 0.0, 0.0) distance [sj_band]
  18  contact model smoothjoint range group 'grupo_foliacao'
  19  ; Rigidez do plano = rigidez da matriz; resistencias do Main
  20  contact property dip [folia_dip] ddir [folia_ddir] sj_kn [p_fjkn] sj_ks [p_fjks] sj_coh [p_sjcoh] sj_ten [p_sjten] sj_fa [p_sjfa] sj_fric [p_sjfric] range group 'grupo_foli

The plane is described by `dip` (from horizontal) and a dip direction of 90 degrees. The specimen axis is `z`,
so the normal of the plane is `(sin(dip), 0, cos(dip))`. Notebook 03 explains the contact law.

## Step 3 - `03_Confinement_Servo.dat`: bring the sample to the confining stress

A servo moves the lateral wall and the platens, with a velocity proportional to the stress error and limited to
`servo_vmax`. It runs every 20 cycles.

Block 4: the servo law.

In [5]:
show("03_Confinement_Servo.dat", 36, 70)

  36      ; Executa o servo apenas a cada 20 ciclos (evita excesso de processamento)
  37      if global.cycle % 20 # 0 then
  38          exit
  39      endif
  40  
  41      local m       = measure.find(1)
  42      ; [v1] Tensao radial = media de xx e yy (antes so xx). [v3.2] Eixo axial = z
  43      local sigma_r = 0.5 * (measure.stress.xx(m) + measure.stress.yy(m))
  44      local sigma_a = measure.stress.zz(m)
  45  
  46      ; [v1] Progresso no console a cada 5000 ciclos (ciclo, passo de tempo, tensoes em MPa)
  47      if global.cycle % 5000 = 0 then
  48          io.out('ciclo = ' + string(global.cycle) + '  dt = ' + string(global.timestep) + '  radial = ' + string(sigma_r / 1.0e6) + '  axial = ' + string(sigma_a / 1.0e6))
  49      endif
  50  
  51      ; [v1] Velocidade limitada em +-servo_vmax
  52      local v_corr = servo_gain * (servo_target_stress - sigma_r)
  53      if v_corr > servo_vmax then
  54          v_corr = servo_vmax
  55      endif
  56      if v_corr < 

Block 5: when to stop. Both the radial and the axial stress must stay within the tolerance for 50 checks in a row.

In [6]:
show("03_Confinement_Servo.dat", 103, 127)

 103  ; [v1] Criterio de parada do adensamento (antes: 50000 ciclos fixos, sem checar a tensao)
 104  fish define halt_conf
 105      halt_conf = 0
 106      if global.cycle % 20 # 0 then
 107          exit
 108      endif
 109      local m  = measure.find(1)
 110      local sr = 0.5 * (measure.stress.xx(m) + measure.stress.yy(m))
 111      local sa = measure.stress.zz(m)
 112      local er = math.abs(sr - servo_target_stress) / math.abs(servo_target_stress)
 113      local ea = math.abs(sa - servo_target_stress) / math.abs(servo_target_stress)
 114      ; [v2] As duas tensoes (radial e axial) precisam estar dentro da tolerancia. Antes, com "and",
 115      ; o ensaio parou com sigma axial = 4.6 MPa para alvo de 6 MPa (23 % de erro).
 116      conf_cont = conf_cont + 1
 117      if er >= conf_tol then
 118          conf_cont = 0
 119      endif
 120      if ea >= conf_tol then
 121          conf_cont = 0
 122      endif
 123      if conf_cont >= 50 then
 124          halt_conf = 1
 125

## Step 4 - `04_Axial_Loading.dat`: loading and stop rules

The platens move at constant speed (`vel_placa`). Virtual extensometers measure strain like in the lab, and histories
are saved every 500 cycles. The lateral servo keeps the confining stress during loading.

Block 6: the stop rule. The first peak is **locked** when the deviator falls below a fraction of its maximum, so a
later hardening does not move the reported peak.

In [7]:
show("04_Axial_Loading.dat", 441, 470)

 441  ; [v3] O 1o pico fica TRAVADO: quando o desviador cai abaixo de queda_pico x maximo, sig_pico e eps_pico
 442  ; param de subir, mesmo que o pos-pico volte a endurecer.
 443  ; Roda so a cada 500 ciclos (o mesmo intervalo dos historicos)
 444  fish define halt_axial
 445      local ea = 0.0
 446      local er = 0.0
 447      local dv = 0.0
 448      local ep = 0.0
 449      local m  = 0
 450      local d  = 0
 451      halt_axial = 0
 452      if global.cycle % 500 # 0 then
 453          exit
 454      endif
 455      ea = deformacao_axial_macro
 456      er = deformacao_radial_macro
 457      dv = tensao_desv_macro
 458      ep = deformacao_axial_placa
 459      ; Progresso no console a cada 100000 ciclos
 460      if global.cycle % 100000 = 0 then
 461          io.out('ciclo = ' + string(global.cycle) + '  eps_ax ext (%) = ' + string(ea) + '  eps_ax pratos (%) = ' + string(ep) + '  desviador (MPa) = ' + string(dv))
 462      endif
 463      ; [v3] Acompanha o maximo ate travar 

## After the run - `00_processa_resultados.py`

The Python script reads the PFC console log and histories, rebuilds the curves, and compares with the test curves.
Notebook 04 shows the main quantities on synthetic curves.

## Before the run - `00_gera_params.py`

It reads the test table and writes one params file per specimen, including the plane cohesion and friction.
Notebooks 01 and 02 show the mechanics behind it.